# E1 · W4 · Kaggle run 06 — kiểm chứng chiều lệch nhãn bằng dự đoán baseline A (CPU)
**Mục đích:** so IoU giữa dự đoán của baseline A (ảnh val, chưa dùng để train) với nhãn gốc mục i và mục i+1, theo từng video; kết luận theo luật đặt trước trong ADR-0006 (issue #36). Không train, không dùng test.

**Trước khi chạy:**
1. Dataset `e1-w3-tools` → **New Version** thêm `w4_offset_check.py` (giữ `w3_export_yolo.py`, `w3_analyze_baseline.py`, `annotation_audit.csv`).
2. *Add Input*: `e1-svanstrom-raw`, `e1-w3-tools` (bản mới), output notebook `e1-w3-run03b` (bản có `e1-yolo-v1.zip`, lấy `yolo_manifest.csv.gz`), output notebook `e1-w3-run04` (lấy `predictions.json`).
3. *Settings*: Accelerator **None**, Internet **On**. Điền `COMMIT`.

**Sau khi chạy:** tải `w4_offset_check.zip` → giải nén vào thư mục kết quả run 06.

In [ ]:
COMMIT = ""   # <-- SHA day du 40 ky tu

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install mcos-decoder
import mcos_decoder; print("mcos-decoder OK")

In [ ]:
# Tim thu muc du lieu tho; neu dataset tho la file .zip thi giai nen vao /kaggle/working/raw
if not glob.glob("/kaggle/input/**/Data/Video_V", recursive=True) and not glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True):
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        print("giai nen", z); zipfile.ZipFile(z).extractall("/kaggle/working/raw")
DATA_ROOT = "/kaggle/working/raw" if glob.glob("/kaggle/working/raw/**/Data/Video_V", recursive=True) else "/kaggle/input"
assert glob.glob(f"{DATA_ROOT}/**/Data/Video_V", recursive=True), "Khong thay Data/Video_V — kiem tra input e1-svanstrom-raw"
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
SCRIPT = find("w4_offset_check.py")
for n in ["w3_export_yolo.py", "w3_analyze_baseline.py"]:
    assert os.path.dirname(find(n)) == os.path.dirname(SCRIPT), f"{n} phai nam cung thu muc voi w4_offset_check.py"
MANIFEST = find("yolo_manifest.csv.gz"); PRED = find("predictions.json"); AUDIT = find("annotation_audit.csv")
for p in (SCRIPT, MANIFEST, PRED, AUDIT):
    print(f"sha256={sha(p)[:16]}  {p}")

In [ ]:
!{sys.executable} {SCRIPT} --data-root {DATA_ROOT} --manifest {MANIFEST} --predictions {PRED} --audit {AUDIT} --out /kaggle/working --commit {COMMIT}

## Tự kiểm tra (phải in `PASS`)

In [ ]:
out = "/kaggle/working/w4_offset_check"
st = json.load(open(f"{out}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
s = json.load(open(f"{out}/offset_check_summary.json"))
print(json.dumps(s["groups"], indent=1)); print("control_ok:", s["control_ok"]); print("verdict:", s["verdict"])
print("PASS" if all(v["status"] == "OK" for v in st.values()) and len(st) == 4 else "FAIL - gui stage_status.json cho Claude")
p = "/kaggle/working/w4_offset_check.zip"; print(os.path.getsize(p) // 1024, "KB  sha256", sha(p))